In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np

# X_train, X_test are image data (N, C, H, W), y_train, y_test are age labels
# Images are converted to float32 for model input
# Labels are converted to float32 and reshaped to (N, 1) for regression task
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train).float().view(-1, 1)
y_test_tensor = torch.from_numpy(y_test).float().view(-1, 1)

print(f"X_train_tensor shape: {X_train_tensor.shape}")
print(f"X_test_tensor shape: {X_test_tensor.shape}")
print(f"y_train_tensor shape: {y_train_tensor.shape}")
print(f"y_test_tensor shape: {y_test_tensor.shape}")

In [ ]:
# 2. Create TensorDataset objects

# TensorDataset wraps tensors, providing a way to access corresponding slices of tensors
# along the first dimension.
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")


In [ ]:
# 3. Create DataLoaders


# DataLoader provides an iterable over the dataset, supporting batching, shuffling, and multiprocessing.
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Number of batches in train_loader: {len(train_loader)}")
print(f"Number of batches in test_loader: {len(test_loader)}")

In [ ]:
# 4. Print shape of one batch

# Get one batch to verify its shape
for images, labels in train_loader:
    print(f"Shape of one batch of images: {images.shape}") # Expected: [batch_size, channels, height, width]
    print(f"Shape of one batch of labels: {labels.shape}") # Expected: [batch_size, 1]
    break # Only get the first batch

In [ ]:
# 5. Display sample images

# Get a batch of training data to visualize
dataiter = iter(train_loader)
images, labels = next(dataiter)

# Convert images to numpy for matplotlib display.
# PyTorch images are typically (N, C, H, W). Matplotlib expects (H, W, C) for color images.
# Transpose dimensions from (C, H, W) to (H, W, C) for correct display.
images_np = images.numpy().transpose(0, 2, 3, 1)

# Create a figure and a set of subplots to display 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 3))

# Display 5 sample images with their age labels
for i in range(5):
    ax = axes[i]
    ax.imshow(images_np[i])
    ax.set_title(f"Age: {int(labels[i].item())}") # Display age as an integer
    ax.axis('off') # Hide axes for cleaner visualization

plt.suptitle("Sample Images from Training Data", fontsize=16)
plt.show()

In [ ]:
# Task 1: Write your model class here:

class AgePredictorCNN(nn.Module):
    def __init__(self):
        super(AgePredictorCNN, self).__init__()
        # Convolutional Block 1
        # Input: 3x64x64 (RGB image)
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1) # Output: 16x64x64
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)      # Output: 16x32x32

        # Convolutional Block 2
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1) # Output: 32x32x32
        # After pooling: 32x16x16

        # Convolutional Block 3
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1) # Output: 64x16x16
        # After pooling: 64x8x8

        # Convolutional Block 4
        self.conv4 = nn.Conv2d(64, 128, kernel_size=3, padding=1) # Output: 128x8x8
        # After pooling: 128x4x4 (THIS IS THE ERROR. Should be 128x2x2 for 36x36 input)

        # Calculate the size of the flattened output from the convolutional layers
        # Original calculation 128 channels * 4 (height) * 4 (width) was for 64x64 input
        # For 36x36 input and 4 MaxPool2d(2,2) layers:
        # 36 -> 18 -> 9 -> 4 -> 2
        # So, it should be 128 * 2 * 2 = 512
        self.fc_input_size = 128 * 2 * 2  # Corrected from 128 * 4 * 4

        # Fully Connected (Linear) Layers for regression prediction
        # These are the 4 linear layers requested.
        self.fc1 = nn.Linear(self.fc_input_size, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 1) # Output a single scalar value (age)

    def forward(self, x):
        # Apply convolutional layers followed by ReLU activation and Max Pooling
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.pool(F.relu(self.conv4(x)))

        # Flatten the feature maps for input to the fully connected layers
        x = x.view(-1, self.fc_input_size) # -1 infers the batch size dynamically

        # Apply fully connected layers with ReLU activation for hidden layers
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x) # Output layer has no activation function for regression

        return x

In [ ]:
# Task 2: Write your training loop here:

def train_epoch(model, dataloader, loss_fn, optimizer, device):
    model.train() # Set the model to training mode
    total_loss = 0
    # Iterate over batches in the dataloader
    for batch_idx, (images, labels) in enumerate(dataloader):
        images, labels = images.to(device), labels.to(device) # Move data to the specified device

        # Zero the gradients accumulated from the previous iteration
        optimizer.zero_grad()

        # Forward pass: Compute predicted outputs by passing inputs to the model
        outputs = model(images)

        # Calculate the loss between predicted outputs and actual labels
        loss = loss_fn(outputs, labels)

        # Backward pass: Compute gradient of the loss with respect to model parameters
        loss.backward()

        # Update model parameters using the calculated gradients
        optimizer.step()

        total_loss += loss.item() # Accumulate the loss

    # Return the average loss for the epoch
    return total_loss / len(dataloader)

In [ ]:
# Task 3: Write your validation loop here:

def validate_epoch(model, dataloader, loss_fn, device):
    model.eval() # Set the model to evaluation mode (disables dropout, batch normalization, etc.)
    total_loss = 0
    # Disable gradient calculation, as it's not needed for validation
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device) # Move data to device

            # Forward pass
            outputs = model(images)

            # Calculate loss
            loss = loss_fn(outputs, labels)
            total_loss += loss.item() # Accumulate the loss

    # Return the average loss for the epoch
    return total_loss / len(dataloader)

In [ ]:
# Task 4: Define device, model, loss, optimizer:

# Set the device to GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Instantiate the AgePredictorCNN model and move it to the selected device
model = AgePredictorCNN().to(device)
print(model)

# Define the loss function: Mean Squared Error (MSE) is suitable for regression tasks
loss_fn = nn.MSELoss()

# Define the optimizer: Adam is a popular choice for its efficiency
learning_rate = 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# Task 5: Start training for 20 epochs:

num_epochs = 20 # Define the total number of training epochs

# Lists to store loss values for plotting later
train_losses = []
val_losses = []

print("Starting training...")
for epoch in range(num_epochs):
    # Perform one training epoch and get the average training loss
    avg_train_loss = train_epoch(model, train_loader, loss_fn, optimizer, device)

    # Perform one validation epoch and get the average validation loss
    avg_val_loss = validate_epoch(model, test_loader, loss_fn, device)

    # Store the losses
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    # Print epoch-wise statistics
    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}")

print("Training complete!")

In [ ]:
# Task 1: Write your code here:

# Task 1: Plot the training and validation loss over epochs
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Task 2 (Bonus): Write your code here:

model.eval() # Set the model to evaluation mode before making predictions
fig, axes = plt.subplots(1, 5, figsize=(20, 4)) # Create a figure with 5 subplots

# Get a batch of images and labels from the test loader
dataiter = iter(test_loader)
images, actual_labels = next(dataiter)

# Move images and labels to the same device as the model
images = images.to(device)
actual_labels = actual_labels.to(device)

# Make predictions with the trained model without calculating gradients
with torch.no_grad():
    predicted_labels = model(images)

# Convert images back to numpy for matplotlib display (transpose dimensions)
images_np = images.cpu().numpy().transpose(0, 2, 3, 1)

# Convert actual and predicted labels to numpy for display
actual_labels_np = actual_labels.cpu().numpy()
predicted_labels_np = predicted_labels.cpu().numpy()

# Display 5 sample images with their actual and predicted ages
for i in range(5):
    ax = axes[i]
    ax.imshow(images_np[i])
    ax.set_title(f"Actual: {int(actual_labels_np[i].item())}\nPred: {predicted_labels_np[i].item():.1f}")
    ax.axis('off')

plt.suptitle("Sample Predictions (Actual vs. Predicted Age)", fontsize=16)
plt.show()